In [23]:
import os
import sys
import pandas as pd

# `classifier` 모듈을 임포트하기 위해 프로젝트 루트를 경로에 추가합니다.
# 이 노트북이 `notebooks` 디렉토리 안에 있다고 가정합니다.
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from notebooks.classifier import SigLIPClassifier

# SigLIP Batch Classification & CSV Export

이 노트북은 SigLIP 모델을 사용하여 배치 이미지 분류를 수행하고 결과를 CSV로 저장합니다.

## 1. 모델 설정

In [51]:
# 학습된 모델 가중치 경로 (파인튜닝된 모델 사용 시)
CHECKPOINT_PATH = "../models/checkpoints/siglip-base-patch16-224/best_model_siglip-base-patch16-224_20251202_004235.pth"
# CHECKPOINT_PATH = "../models/checkpoints/siglip2-base-patch16-224/best_model_siglip2-base-patch16-224_20251202_024457.pth"

# 또는 기본 HF 모델 사용 시 (CHECKPOINT_PATH = None)
# CHECKPOINT_PATH = None

# 학습 시 사용한 기본 모델 ID
MODEL_ID = "google/siglip-base-patch16-224"
# MODEL_ID = "google/siglip2-base-patch16-224"

# 분류기 초기화
classifier = SigLIPClassifier(
    model_id=MODEL_ID,
    checkpoint_path=CHECKPOINT_PATH,
    threshold=THRESHOLD
)

모델 로딩 중...


Some weights of SiglipForImageClassification were not initialized from the model checkpoint at google/siglip-base-patch16-224 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


파인튜닝된 가중치 로딩: ../models/checkpoints/siglip-base-patch16-224/best_model_siglip-base-patch16-224_20251202_004235.pth
모델 가중치 로드 완료.
모델이 cuda 디바이스로 이동되었습니다.


## 2. 테스트 이미지 로드

In [52]:
# 테스트 이미지가 있는 폴더 경로
test_folder = '../../test_images/images/'

# 폴더 내 모든 이미지 파일 찾기
test_paths = []
for filename in os.listdir(test_folder):
    if filename.endswith(('.jpg', '.jpeg', '.png')):
        full_path = os.path.join(test_folder, filename)
        test_paths.append(full_path)

print(f"총 {len(test_paths)}개의 테스트 이미지를 찾았습니다.")

총 107개의 테스트 이미지를 찾았습니다.


## 3. 배치 분류 실행

In [53]:
# 분류 임계값 (Sigmoid threshold)
THRESHOLD = 0.8

# 배치 분류 수행
results = classifier.batch_classify(test_paths, threshold=THRESHOLD)

처리 중: 1/107 - IMG_8704.jpg
처리 중: 2/107 - 실내이미지1.jpg
처리 중: 3/107 - 8.png
처리 중: 4/107 - KakaoTalk_Photo_2025-09-17-19-27-03.jpg
처리 중: 5/107 - 33.png
처리 중: 6/107 - IMG_6552.jpeg
처리 중: 7/107 - IMG_7110.jpg
처리 중: 8/107 - 12.jpeg
처리 중: 9/107 - 20221009051030_photo4_VQqJSCTzFQ7q.jpeg
처리 중: 10/107 - IMG_9034.jpg
처리 중: 11/107 - 34.png
처리 중: 12/107 - 19.png
처리 중: 13/107 - 43.jpg
처리 중: 14/107 - 6.png
처리 중: 15/107 - KakaoTalk_Photo_2025-09-17-19-28-16.jpg
처리 중: 16/107 - IMG_6545.jpeg
처리 중: 17/107 - IMG_6539.jpg
처리 중: 18/107 - 45.png
처리 중: 19/107 - 31.png
처리 중: 20/107 - IMG_8702.jpg
처리 중: 21/107 - IMG_6547.jpeg
처리 중: 22/107 - 사진4.jpg
처리 중: 23/107 - 9.png
처리 중: 24/107 - 39.png
처리 중: 25/107 - KakaoTalk_Photo_2025-09-17-19-30-12.jpg
처리 중: 26/107 - gn_b1.jpeg
처리 중: 27/107 - 41.png
처리 중: 28/107 - 실내사진3.jpg
처리 중: 29/107 - test4_패스트푸드.jpg
처리 중: 30/107 - 20191006074314107_photo_ChOa61ijX3df.jpeg
처리 중: 31/107 - 15.png
처리 중: 32/107 - IMG_6550.jpeg
처리 중: 33/107 - 실내사진1.jpg
처리 중: 34/107 - 1.png
처리 중: 35/107 - 

## 4. 결과 미리보기

In [54]:
# 첫 5개 결과 확인
for i, result in enumerate(results[:5]):
    print(f"\n이미지 {i+1}: {os.path.basename(result['image_path'])}")
    for key, value in result.items():
        if key != 'image_path' and not key.endswith('_prob'):
            prob = result.get(f"{key}_prob", 0.0)
            status = "✓" if value else "✗"
            print(f"  {status} {key}: {value} (확률: {prob:.3f})")


이미지 1: IMG_8704.jpg
  ✗ has_step: False (확률: 0.160)
  ✓ has_movable_chair: True (확률: 0.909)
  ✗ has_high_chair: False (확률: 0.172)
  ✗ has_fixed_chair: False (확률: 0.390)
  ✗ has_floor_chair: False (확률: 0.147)

이미지 2: 실내이미지1.jpg
  ✗ has_step: False (확률: 0.165)
  ✓ has_movable_chair: True (확률: 0.906)
  ✗ has_high_chair: False (확률: 0.383)
  ✗ has_fixed_chair: False (확률: 0.409)
  ✗ has_floor_chair: False (확률: 0.183)

이미지 3: 8.png
  ✗ has_step: False (확률: 0.323)
  ✗ has_movable_chair: False (확률: 0.717)
  ✗ has_high_chair: False (확률: 0.164)
  ✓ has_fixed_chair: True (확률: 0.817)
  ✗ has_floor_chair: False (확률: 0.282)

이미지 4: KakaoTalk_Photo_2025-09-17-19-27-03.jpg
  ✗ has_step: False (확률: 0.609)
  ✓ has_movable_chair: True (확률: 0.920)
  ✗ has_high_chair: False (확률: 0.297)
  ✗ has_fixed_chair: False (확률: 0.631)
  ✗ has_floor_chair: False (확률: 0.381)

이미지 5: 33.png
  ✗ has_step: False (확률: 0.271)
  ✗ has_movable_chair: False (확률: 0.520)
  ✗ has_high_chair: False (확률: 0.717)
  ✗ has_fixed_chair:

## 5. CSV로 저장

In [55]:
# DataFrame으로 변환
df = pd.DataFrame(results)

# 파일명만 추출 (전체 경로 대신)
df['file_path'] = df['image_path'].apply(os.path.basename)

# image_path 열 제거 (file_path로 대체)
df = df.drop('image_path', axis=1)

# 열 순서 재정렬: file_path를 맨 앞으로
cols = ['file_path'] + [col for col in df.columns if col != 'file_path']
df = df[cols]

# 결과 저장 경로
OUTPUT_DIR = "../outputs/predictions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 모델 타입 자동 감지 (SigLIP vs SigLIP2)
if 'siglip2' in MODEL_ID.lower():
    model_type = 'siglip2'
else:
    model_type = 'siglip'

# 파일명 생성 (모델 타입 + fine-tuned/base)
if CHECKPOINT_PATH:
    model_name = f"{model_type}_ft"  # fine-tuned
else:
    model_name = f"{model_type}_base"  # base model

SAVE_PATH = os.path.join(OUTPUT_DIR, f'{model_name}_{THRESHOLD}.csv')

# CSV 저장
df.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')

print(f"\n결과가 저장되었습니다: {SAVE_PATH}")
print(f"총 {len(df)}개 이미지 분류 완료")
print(f"사용된 모델: {model_name} (threshold={THRESHOLD})")


결과가 저장되었습니다: ../outputs/predictions/siglip_ft_0.8.csv
총 107개 이미지 분류 완료
사용된 모델: siglip_ft (threshold=0.8)


## 6. 통계 확인

In [47]:
# 레이블별 감지 횟수
label_cols = ['has_step', 'has_movable_chair', 'has_high_chair', 'has_fixed_chair', 'has_floor_chair']

print("\n=== 레이블별 감지 통계 ===")
for label in label_cols:
    count = df[label].sum()
    ratio = count / len(df) * 100
    print(f"{label}: {count}/{len(df)} ({ratio:.1f}%)")


=== 레이블별 감지 통계 ===
has_step: 0/107 (0.0%)
has_movable_chair: 60/107 (56.1%)
has_high_chair: 2/107 (1.9%)
has_fixed_chair: 17/107 (15.9%)
has_floor_chair: 1/107 (0.9%)
